# Chapter 7, Guide 1: Choosing and testing the Trotter parameters

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 7,
"Choosing the Trotter parameters" and "Testing a Trotter circuit".

The guide, in order: decide the splitting by grouping terms that commute internally
(only the number of *groups* enters the error); choose $r$ from the target accuracy,
$r\approx\Lambda^2t^2/\epsilon$ (first order) or $r\approx(\Lambda^3t^3/\epsilon)^{1/2}$
(symmetric), $\Lambda=\sum_k\|H_k\|$; always use the symmetric formula and merge the
shared half-steps; then test slopes on a single step (2 and 3) and on the whole
$r$-step circuit at fixed $t$ (1 and 2).

The model is the transverse-field Ising chain
$H = -J\sum_i Z_iZ_{i+1} - h\sum_i X_i$ on $n=4$ qubits, open boundary.
`pauli_evolution` is the routine of Guide 4 (notebook 07d), copied here so the
notebook stands alone.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, SparsePauliOp

bq = lambda j, tot: tot - 1 - j


def pauli_evolution(circuit, pauli_string, angle, qubits):
    active = [k for k, p in enumerate(pauli_string) if p != "I"]
    if not active:
        return
    for k in active:
        if pauli_string[k] == "X": circuit.h(qubits[k])
        elif pauli_string[k] == "Y": circuit.sdg(qubits[k]); circuit.h(qubits[k])
    last = qubits[active[-1]]
    for k in active[:-1]: circuit.cx(qubits[k], last)
    circuit.rz(2 * angle, last)
    for k in reversed(active[:-1]): circuit.cx(qubits[k], last)
    for k in active:
        if pauli_string[k] == "X": circuit.h(qubits[k])
        elif pauli_string[k] == "Y": circuit.h(qubits[k]); circuit.s(qubits[k])

## Step 1: the splitting

All $ZZ$ terms commute with each other, all $X$ terms commute with each other. Two
groups, $H_Z$ and $H_X$, however long the chain.

In [2]:
n, J, h = 4, 1.0, 0.8
q = [bq(j, n) for j in range(n)]


def zz_string(i):
    s = ["I"] * n; s[i] = "Z"; s[i + 1] = "Z"; return "".join(s)


def x_string(i):
    s = ["I"] * n; s[i] = "X"; return "".join(s)


group_Z = [(zz_string(i), -J) for i in range(n - 1)]
group_X = [(x_string(i), -h) for i in range(n)]
terms = group_Z + group_X
H = SparsePauliOp.from_list(terms).to_matrix()
HZ = SparsePauliOp.from_list(group_Z).to_matrix()
HX = SparsePauliOp.from_list(group_X).to_matrix()
print("terms:", terms)
print("[H_Z, H_X] = 0?", np.allclose(HZ @ HX, HX @ HZ), " (they do not commute, so Trotter error is real)")

terms: [('ZZII', -1.0), ('IZZI', -1.0), ('IIZZ', -1.0), ('XIII', -0.8), ('IXII', -0.8), ('IIXI', -0.8), ('IIIX', -0.8)]
[H_Z, H_X] = 0? False  (they do not commute, so Trotter error is real)


## Step 2: choosing $r$ from the bound

$\Lambda = \|H_Z\| + \|H_X\|$ with the operator (spectral) norm.

In [3]:
Lambda = np.linalg.norm(HZ, 2) + np.linalg.norm(HX, 2)
t, eps = 2.0, 1e-2
r1 = int(np.ceil(Lambda**2 * t**2 / eps))
r2 = int(np.ceil(np.sqrt(Lambda**3 * t**3 / eps)))
print(f"Lambda = {Lambda:.3f}, t = {t}, target eps = {eps}")
print(f"first-order bound: r ≈ {r1}   symmetric bound: r ≈ {r2}")

Lambda = 6.200, t = 2.0, target eps = 0.01
first-order bound: r ≈ 15377   symmetric bound: r ≈ 437


## The circuits: first order, symmetric, and symmetric with merged half-steps

In [4]:
def apply_group(qc, group, dt):
    for P, c in group:
        pauli_evolution(qc, P, c * dt, q)


def trotter_circuit(t, r, order):
    """order = 1: (e^{-iH_Z dt} e^{-iH_X dt})^r ; order = 2: symmetric with merged halves."""
    dt = t / r
    qc = QuantumCircuit(n)
    if order == 1:
        for _ in range(r):
            apply_group(qc, group_Z, dt); apply_group(qc, group_X, dt)
    else:
        # merged: Z(dt/2) [X(dt) Z(dt)]^{r-1} X(dt) Z(dt/2)  -> r+1 Z blocks instead of 2r
        apply_group(qc, group_Z, dt / 2)
        for k in range(r):
            apply_group(qc, group_X, dt)
            apply_group(qc, group_Z, dt if k < r - 1 else dt / 2)
    return qc


def error(t, r, order):
    return np.linalg.norm(Operator(trotter_circuit(t, r, order)).data - expm(-1j * H * t), 2)


# the merged symmetric circuit equals the unmerged one exactly
def symmetric_unmerged(t, r):
    dt = t / r; qc = QuantumCircuit(n)
    for _ in range(r):
        apply_group(qc, group_Z, dt / 2); apply_group(qc, group_X, dt); apply_group(qc, group_Z, dt / 2)
    return qc


assert np.allclose(Operator(trotter_circuit(1.0, 5, 2)).data, Operator(symmetric_unmerged(1.0, 5)).data)
print("merged and unmerged symmetric circuits agree; blocks:",
      "merged", 2 * 5 + 1, "vs unmerged", 3 * 5)

merged and unmerged symmetric circuits agree; blocks: merged 11 vs unmerged 15


## Test 1: single-step slopes, 2 for first order and 3 for symmetric

In [5]:
dts = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
e1 = [error(dt, 1, 1) for dt in dts]
e2 = [error(dt, 1, 2) for dt in dts]
s1 = np.polyfit(np.log(dts), np.log(e1), 1)[0]; s2 = np.polyfit(np.log(dts), np.log(e2), 1)[0]
print(f"single step: first-order slope {s1:.2f} (expect 2), symmetric slope {s2:.2f} (expect 3)")
assert abs(s1 - 2) < 0.1 and abs(s2 - 3) < 0.15

single step: first-order slope 1.97 (expect 2), symmetric slope 2.98 (expect 3)


## Test 2: whole circuit at fixed $t$, slopes drop by one to 1 and 2

In [6]:
rs = np.array([4, 8, 16, 32, 64])
dts_r = t / rs
E1 = [error(t, r, 1) for r in rs]; E2 = [error(t, r, 2) for r in rs]
S1 = np.polyfit(np.log(dts_r), np.log(E1), 1)[0]; S2 = np.polyfit(np.log(dts_r), np.log(E2), 1)[0]
print(f"fixed t={t}: first-order slope {S1:.2f} (expect 1), symmetric slope {S2:.2f} (expect 2)")
assert abs(S1 - 1) < 0.15 and abs(S2 - 2) < 0.15
plt.loglog(dts_r, E1, "o-", label="first order"); plt.loglog(dts_r, E2, "s-", label="symmetric")
plt.xlabel(r"$\Delta t = t/r$"); plt.ylabel("error at fixed t"); plt.legend(); plt.show()

fixed t=2.0: first-order slope 1.09 (expect 1), symmetric slope 2.03 (expect 2)


## The bound is pessimistic: halve $r$ until the answer visibly changes

Compare the $r$ the bound asks for against the $r$ that actually reaches $\epsilon$.

In [7]:
print(f"target eps = {eps}")
for order, r_bound in [(1, r1), (2, r2)]:
    r = r_bound
    while r > 1 and error(t, r // 2, order) < eps:
        r //= 2
    print(f"order {order}: bound says r = {r_bound}, error at r={r} is {error(t, r, order):.2e}, "
          f"at r={max(r//2,1)} it is {error(t, max(r//2,1), order):.2e}")

target eps = 0.01


order 1: bound says r = 15377, error at r=480 is 5.30e-03, at r=240 it is 1.06e-02


order 2: bound says r = 437, error at r=27 is 7.75e-03, at r=13 it is 3.36e-02


## Cost of one symmetric step on hardware

In [8]:
tc = transpile(trotter_circuit(t, 1, 2), basis_gates=["u", "cx"], optimization_level=1)
print("one merged symmetric step:", tc.count_ops())

one merged symmetric step: OrderedDict({'cx': 12, 'u': 10})
